# Task 2: Real Estate Housing Price Prediction (Ridge & Lasso Regression)

**Objective:** Build and compare Ridge Regression and Lasso Regression models to predict house values from housing features.

This notebook covers dataset loading, data inspection, exploratory data analysis, an 80/20 train-test split, feature scaling, model training, evaluation using MAE/MSE/RMSE/R², and visualizations.

**Dataset note:** This notebook uses Scikit-learn's California Housing dataset as a reproducible sample dataset. If the internship portal provides a required housing CSV/starter dataset, use that dataset instead and update the dataset-loading cell.

## 1. Import the required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load the housing dataset

In [ ]:
# Scikit-learn downloads the California Housing dataset the first time it is used.
housing = fetch_california_housing(as_frame=True)
X = housing.data.copy()
y = housing.target.copy()

df = X.copy()
df["MedHouseValue"] = y

print("Dataset shape:", df.shape)
display(df.head())
print("\nFeature names:")
print(list(X.columns))
print("\nTarget description:", housing.target_names[0] if housing.target_names else "Median house value")

## 3. Inspect and clean the data

In [ ]:
print("Missing values in each column:")
display(df.isnull().sum())

print("\nSummary statistics:")
display(df.describe().round(2))

# Remove duplicate rows if any are present.
df = df.drop_duplicates().copy()
X = df.drop(columns=["MedHouseValue"])
y = df["MedHouseValue"]
print("Shape after removing duplicates:", df.shape)

## 4. Exploratory data analysis

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(y, bins=40, kde=True)
plt.title("Distribution of Median House Values")
plt.xlabel("Median house value (units of $100,000)")
plt.show()

plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## 5. Split data into training and testing sets (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 6. Build and train Ridge and Lasso Regression models

Both models use a pipeline with `StandardScaler`. Scaling is fitted only on the training data to prevent data leakage. Ridge uses L2 regularization; Lasso uses L1 regularization.

In [ ]:
ridge_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
lasso_model = make_pipeline(StandardScaler(), Lasso(alpha=0.01, max_iter=10000))

ridge_model.fit(X_train, y_train)
lasso_model.fit(X_train, y_train)

ridge_pred = ridge_model.predict(X_test)
lasso_pred = lasso_model.predict(X_test)

print("Ridge and Lasso models trained successfully.")

## 7. Evaluate model performance

In [ ]:
def regression_metrics(model_name, actual, predicted):
    mse = mean_squared_error(actual, predicted)
    return {
        "Model": model_name,
        "MAE": mean_absolute_error(actual, predicted),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2 Score": r2_score(actual, predicted)
    }

comparison = pd.DataFrame([
    regression_metrics("Ridge Regression", y_test, ridge_pred),
    regression_metrics("Lasso Regression", y_test, lasso_pred)
])
display(comparison.round(4))

print("Lower MAE, MSE and RMSE are better. Higher R2 is better.")

## 8. Actual vs. predicted values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, predictions, title in [
    (axes[0], ridge_pred, "Ridge Regression"),
    (axes[1], lasso_pred, "Lasso Regression")
]:
    ax.scatter(y_test, predictions, alpha=0.35)
    low = min(y_test.min(), predictions.min())
    high = max(y_test.max(), predictions.max())
    ax.plot([low, high], [low, high], "r--")
    ax.set_xlabel("Actual house value")
    ax.set_ylabel("Predicted house value")
    ax.set_title(title)

plt.tight_layout()
plt.show()

## 9. Residual analysis

In [ ]:
ridge_residuals = y_test - ridge_pred
lasso_residuals = y_test - lasso_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(ridge_residuals, kde=True, ax=axes[0])
axes[0].set_title("Ridge Residual Distribution")
axes[0].set_xlabel("Actual - Predicted")
sns.histplot(lasso_residuals, kde=True, ax=axes[1])
axes[1].set_title("Lasso Residual Distribution")
axes[1].set_xlabel("Actual - Predicted")
plt.tight_layout()
plt.show()

## 10. Compare model coefficients

In [ ]:
ridge_coefficients = ridge_model.named_steps["ridge"].coef_
lasso_coefficients = lasso_model.named_steps["lasso"].coef_

coefficient_comparison = pd.DataFrame({
    "Feature": X.columns,
    "Ridge Coefficient": ridge_coefficients,
    "Lasso Coefficient": lasso_coefficients
})
display(coefficient_comparison.round(4))

coefficient_comparison.set_index("Feature").plot(kind="bar", figsize=(12, 5))
plt.title("Ridge vs Lasso Feature Coefficients")
plt.ylabel("Coefficient (after feature scaling)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 11. Conclusion

In this task, a housing dataset was used to predict median house values. Ridge Regression and Lasso Regression were trained using an 80/20 train-test split and standardized features. Their performance was compared using MAE, MSE, RMSE and R² score. Actual-versus-predicted plots, residual distributions and coefficient comparisons were used to understand the models. The evaluation table indicates which model performed better on this test set.